In [ ]:
%load_ext autoreload
%autoreload 2

### Update with ESCAPE-seq data

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name not in ("CLSBN26", "PepCL_Manuscript"):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

PROJECT_ROOT: /home/pchati/PepCL_Manuscript
DATA_DIR: /home/pchati/PepCL_Manuscript/Data
PROJECT_ROOT in sys.path: True


In [ ]:
# imports
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.log_odds import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# change gpu
# GPU_ID = 0
# GPU_ID = 1
GPU_ID = 2
# GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")
device=f"cuda:{GPU_ID}"

Using GPU: NVIDIA H100 80GB HBM3 (ID: 2)


### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### MS/ESCAPE-seq train and test sets

In [ ]:
# ms train and test sets
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"

# ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0)
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data_remaining_minimal.csv", index_col=0)

ms_val_data=pd.read_csv(f"{datasets_dir}/ms_val_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

In [ ]:
# escape seq sets
escape_seq_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "ESCAPE_seq"
oncogene_mut_wt_df=pd.read_csv(f"{escape_seq_datasets_dir}/oncogene_mut_wt_df.csv", index_col=0)
escapeseq_reported_neoantigens=pd.read_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens.csv", index_col=0)
# escapeseq_reported_neoantigens_expanded=pd.read_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded.csv", index_col=0)
escapeseq_reported_neoantigens_expanded=pd.read_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded_opt.csv", index_col=0)

In [ ]:
# ligand mhc atlas set
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
# ligand_mhc_atlas_cancer_test_set_for_escape_seq=pd.read_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq.csv", index_col=0)

ligand_mhc_atlas_cancer_test_set_for_escape_seq=pd.read_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq_opt.csv", index_col=0)

In [ ]:
# create version without reported neoantigens
oncogene_mut_wt_df.seq=oncogene_mut_wt_df.seq.str.replace("$", "")
escapeseq_reported_neoantigens_expanded.seq=escapeseq_reported_neoantigens_expanded.seq.str.replace("$", "")
oncogene_mut_wt_df=create_seq_allele_column(oncogene_mut_wt_df)
escapeseq_reported_neoantigens_expanded=create_seq_allele_column(escapeseq_reported_neoantigens_expanded)
seq_overlap=df_col_set_intersection(oncogene_mut_wt_df, escapeseq_reported_neoantigens_expanded, "seq", print_output=True, return_output=True)
oncogene_mut_wt_df_cleaned=oncogene_mut_wt_df[~oncogene_mut_wt_df.seq.isin(seq_overlap)]
oncogene_mut_wt_df_cleaned_dedup=oncogene_mut_wt_df_cleaned.drop_duplicates(subset=["seq_allele"])

20


In [ ]:
# merge to create train set
oncogene_mut_wt_df_cleaned_dedup.rename(columns={"escore": "ba_value"}, inplace=True)
oncogene_mut_wt_df_cleaned_dedup["domain_id"]=1
ms_train_data["domain_id"]=0
ms_escape_seq_train_data=pd.concat([ms_train_data, oncogene_mut_wt_df_cleaned_dedup])
ms_escape_seq_train_data.reset_index(inplace=True, drop=True)
print_data_stats(ms_escape_seq_train_data)

Total: 35,987,673
Pos: 731,708 | 2.03%
Neg: 35,255,965 | 97.97%
N alleles: 197


In [ ]:
# remove X containing seqs and A68 sequences
def filter_sequences_without_x(df, seq_col='seq'):
    return df[~df[seq_col].str.contains('X', na=False)].reset_index(drop=True)

ms_escape_seq_train_data_noX=filter_sequences_without_x(ms_escape_seq_train_data)
ms_escape_seq_train_data_noX_noA68=ms_escape_seq_train_data_noX.query("allele != 'A6801'")
ms_escape_seq_train_data_noX_noA68.reset_index(inplace=True, drop=True)
print_data_stats(ms_escape_seq_train_data_noX_noA68)

Total: 35,356,015
Pos: 719,051 | 2.03%
Neg: 34,636,964 | 97.97%
N alleles: 196


In [ ]:
# match alleles that are present in ms (for new alleles, its fine to keep them in)
ms_escape_seq_train_data_noX_noA68["seq_len"]=ms_escape_seq_train_data_noX_noA68.seq.str.replace("$", "").apply(len)
ms_escape_seq_train_data_noX_noA68_matched_allele=ms_escape_seq_train_data_noX_noA68[ms_escape_seq_train_data_noX_noA68.allele.isin(oncogene_mut_wt_df_cleaned_dedup.allele.unique().tolist())].query("seq_len == 9")
ms_escape_seq_train_data_noX_noA68_matched_allele.reset_index(inplace=True, drop=True)
print_data_stats(ms_escape_seq_train_data_noX_noA68_matched_allele)

Total: 12,120,124
Pos: 245,892 | 2.03%
Neg: 11,874,232 | 97.97%
N alleles: 49


In [ ]:
# full process
ms_escape_seq_train_data_noX_noA68_matched_allele=full_process_data(ms_escape_seq_train_data_noX_noA68_matched_allele, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)

escapeseq_reported_neoantigens=full_process_data(escapeseq_reported_neoantigens, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)

escapeseq_reported_neoantigens_expanded=full_process_data(escapeseq_reported_neoantigens_expanded, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)

#### Background datasets

In [ ]:
%%time
# for ranking
# chp_background_10k_per_allele_x_len_processed=pd.read_csv(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.csv.gz")

# E with optimal models
chp_background_10k_per_allele_x_len_processed=pd.read_csv(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed_opt.csv.gz")

CPU times: user 1min 10s, sys: 8.95 s, total: 1min 19s
Wall time: 1min 19s


### Load MHCPrime

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
%%time
# load base model (optionally, can load model_params directly)
# run_name = "MHCPrime_Base"
# run_dir = MODEL_CHECKPOINTS_DIR / run_name
# model_dir = run_dir / "Checkpoints"

# mhcprime=init_model(base_model_params, print_check=False, print_params=True)
# mhcprime.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
# mhcprime.eval()
# print("Model loaded from checkpoint...")
# clear_all_gpu_memory()

# NOTE: original model
# base_model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Base" / "Checkpoints"

# NOTE: must change model here if running ablations below
base_model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Base_for_Ablations" / "Checkpoints"

ckpt_path = base_model_dir / "model_final.pt"

device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

clear_all_gpu_memory()

mhcprime = init_model(
    base_model_params,
    print_check=False,
    print_params=True,
)

state_dict = torch.load(
    ckpt_path,
    map_location="cpu",
)

mhcprime.load_state_dict(state_dict, strict=True)
mhcprime.to(device)
mhcprime.eval()

print("Model loaded from checkpoint...")
print(f"Checkpoint: {ckpt_path}")
print(f"Model device: {next(mhcprime.parameters()).device}")


All GPU memory has been cleared.
Current GPU memory allocated: 0.00 GB
Current GPU memory reserved: 0.00 GB
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
Checkpoint: /home/pchati/PepCL_Manuscript/Model_Checkpoints/MHCPrime_Base_for_Ablations/Checkpoints/model_final.pt
Model device: cuda:2
CPU times: user 446 ms, sys: 64.3 ms, total: 510 ms
Wall time: 531 ms


### PepCL updating

#### Run PepCL

In [ ]:
# for classification based training
pepcl_base_params_cls=dict(
            tokenizer=tokenizer,
            device="cuda",
            steps = 200,
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,
            num_ms_pos = 2000,
            num_nonms_pos = 2000,
            ms_neg_ratio = 1.0,
            nonms_neg_ratio = 1.0,
            auto_calibrate = True,
            calibration_steps = 1,
            ms_batch_size = 3072,
            nonms_batch_size = 3072,
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,
            primary_task_id = 1,
            sequential_backprop = True,
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "logsmoothap",
            nonms_loss_alpha = 10.0,
            ft_lambda = 1.0,
            dr_lambda = 1e-5,
            dr_loss_type = "mse"
)

# for continuous value training
pepcl_base_params_con=dict(
            tokenizer=tokenizer,
            device="cuda",
            steps = 200,
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,
            num_ms_pos = 2000,
            num_nonms_pos = 2000,
            ms_neg_ratio = 1.0,
            nonms_neg_ratio = 1.0,
            auto_calibrate = True,
            calibration_steps = 1,
            ms_batch_size = 3072,
            nonms_batch_size = 3072,
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,
            primary_task_id = 1,
            sequential_backprop = True,
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "spearman",
            nonms_loss_alpha = 0.01,
            ft_lambda = 1.0,
            dr_lambda = 1e-5,
            dr_loss_type = "mse",
            continuous_target_col="ba_value",
            continuous_ft_sampling_mode="allele_aware",
)

In [ ]:
# init models
escape_seq_neoant_ft_cls = copy.deepcopy(mhcprime) # classification
escape_seq_neoant_pepcl_cls = copy.deepcopy(mhcprime) # classification
escape_seq_neoant_ft_con = copy.deepcopy(mhcprime) # continuous value
escape_seq_neoant_pepcl_con = copy.deepcopy(mhcprime) # continuous value
clear_all_gpu_memory()

All GPU memory has been cleared.
Current GPU memory allocated: 0.52 GB
Current GPU memory reserved: 0.56 GB


In [ ]:
# get optimal ft and pepcl params
ft_base_params_cls_optimal=pepcl_base_params_cls.copy()
pepcl_base_params_cls_optimal=pepcl_base_params_cls.copy()

ft_base_params_con_optimal=pepcl_base_params_con.copy()
pepcl_base_params_con_optimal=pepcl_base_params_con.copy()

# steps for both cls versions must be updated to 300, and same with con pepcl, but con ft stays 200 steps
ft_base_params_cls_optimal["steps"]=300
pepcl_base_params_cls_optimal["steps"]=300
pepcl_base_params_con_optimal["steps"]=300

In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_ft_cls,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele,
            use_replay = False,
            # **pepcl_base_params_cls,
            **ft_base_params_cls_optimal,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 300/300 [01:42<00:00,  2.92it/s, MS=0.000000, FT=0.000091, DR=0.000000] 


CPU times: user 1min 56s, sys: 15.1 s, total: 2min 11s
Wall time: 2min 9s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_pepcl_cls,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele,
            use_replay = True,
            # **pepcl_base_params_cls,
            **pepcl_base_params_cls_optimal,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 300/300 [03:56<00:00,  1.27it/s, MS=0.006821, FT=0.000588, DR=0.003010]


CPU times: user 4min, sys: 29 s, total: 4min 29s
Wall time: 4min 25s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_ft_con,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele,
            use_replay = False,
            # **pepcl_base_params_con,
            **ft_base_params_con_optimal,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [01:07<00:00,  2.96it/s, MS=0.000000, FT=0.058702, DR=0.000000]


CPU times: user 1min 19s, sys: 14.1 s, total: 1min 33s
Wall time: 1min 32s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_pepcl_con,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele,
            use_replay = True,
            # **pepcl_base_params_con,
            **pepcl_base_params_con_optimal,

)
training_stats=pepcl.train()

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 9.3 µs


In [ ]:
# copy over negatives
def copy_domain0_negs_to_domain1(df, allele_col="allele", label_col="label", 
                                   seq_col="seq", domain_col="domain_id"):
    d0_negs = df[(df[domain_col] == 0) & (df[label_col] == 0)].copy()
    d0_negs[domain_col] = 1
    return pd.concat([df, d0_negs], ignore_index=True)

ms_escape_seq_train_data_noX_noA68_matched_allele_did0_negs_in_did1=copy_domain0_negs_to_domain1(ms_escape_seq_train_data_noX_noA68_matched_allele, allele_col="allele", label_col="label", seq_col="seq", domain_col="domain_id")
print_data_stats(ms_escape_seq_train_data_noX_noA68_matched_allele_did0_negs_in_did1)

Total: 24,367,624
Pos: 248,092 | 1.02%
Neg: 24,119,532 | 98.98%
N alleles: 49


In [ ]:
# now with the domain id 0 negatives copied over to domain id 1
escape_seq_neoant_ft_w_more_negs_cls = copy.deepcopy(mhcprime) # classification
escape_seq_neoant_pepcl_w_more_negs_cls = copy.deepcopy(mhcprime) # classification

In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_ft_w_more_negs_cls,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele_did0_negs_in_did1,
            use_replay = False,
            # **pepcl_base_params_cls,
            **ft_base_params_cls_optimal,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 300/300 [01:41<00:00,  2.96it/s, MS=0.000000, FT=0.000304, DR=0.000000]


CPU times: user 2min 1s, sys: 19.3 s, total: 2min 21s
Wall time: 2min 19s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_pepcl_w_more_negs_cls,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele_did0_negs_in_did1,
            use_replay = True,
            # **pepcl_base_params_cls,
            **pepcl_base_params_cls_optimal,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 300/300 [03:55<00:00,  1.27it/s, MS=0.006538, FT=0.000517, DR=0.002658]


CPU times: user 4min 10s, sys: 29.2 s, total: 4min 39s
Wall time: 4min 36s


In [ ]:
%%time
"""
Save models
"""
# save_models=True
save_models=False

models_to_save={

    # updated models
    # "escape_seq_neoant_ft_cls": escape_seq_neoant_ft_cls,
    # "escape_seq_neoant_pepcl_cls": escape_seq_neoant_pepcl_cls,
    # "escape_seq_neoant_ft_con": escape_seq_neoant_ft_con,
    # "escape_seq_neoant_pepcl_con": escape_seq_neoant_pepcl_con,

    # "escape_seq_neoant_ft_w_more_negs_cls": escape_seq_neoant_ft_w_more_negs_cls,
    # "escape_seq_neoant_pepcl_w_more_negs_cls": escape_seq_neoant_pepcl_w_more_negs_cls,

    # save optimal ft and pepcl models (sept. 2026)
    "escape_seq_neoant_ft_cls_opt": escape_seq_neoant_ft_cls,
    "escape_seq_neoant_pepcl_cls_opt": escape_seq_neoant_pepcl_cls,
    "escape_seq_neoant_ft_con_opt": escape_seq_neoant_ft_con,
    "escape_seq_neoant_pepcl_con_opt": escape_seq_neoant_pepcl_con,

    "escape_seq_neoant_ft_w_more_negs_cls_opt": escape_seq_neoant_ft_w_more_negs_cls,
    "escape_seq_neoant_pepcl_w_more_negs_cls_opt": escape_seq_neoant_pepcl_w_more_negs_cls,


}

escape_seq_model_checkpoints_dir = MODEL_CHECKPOINTS_DIR / "ESCAPE_seq_Models"
os.makedirs(escape_seq_model_checkpoints_dir, exist_ok=True)

if save_models:
    for model_name, model in models_to_save.items():
        print(model_name)
        torch.save(model.state_dict(), f"{escape_seq_model_checkpoints_dir}/{model_name}.pt")

escape_seq_neoant_ft_cls_opt
escape_seq_neoant_pepcl_cls_opt
escape_seq_neoant_ft_con_opt
escape_seq_neoant_pepcl_con_opt
escape_seq_neoant_ft_w_more_negs_cls_opt
escape_seq_neoant_pepcl_w_more_negs_cls_opt
CPU times: user 435 ms, sys: 476 ms, total: 911 ms
Wall time: 904 ms


#### Load validation selected models for continuous updating experiments

In [ ]:
# get models for scoring
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={
    "mhcprime_abt_rapid": base_model, 
}

for model_file in os.listdir(f"{MODEL_CHECKPOINTS_DIR}/ESCAPE_seq_CON_Validation_Selected_Models_5_Seeds_Cleaned"):
    model_name=model_file.removesuffix(".pt")
    print(model_name)

    ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/ESCAPE_seq_CON_Validation_Selected_Models_5_Seeds_Cleaned/{model_file}"

    device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

    clear_all_gpu_memory()

    model = init_model(
        base_model_params,
        print_check=False,
        print_params=True,
    )

    state_dict = torch.load(
        ckpt_path,
        map_location="cpu",
    )

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    print("Model loaded from checkpoint...")
    print(f"Checkpoint: {ckpt_path}")
    print(f"Model device: {next(model.parameters()).device}")

    base_and_ft_models[model_name]=model

datasplits=[1,2,3,4]
for ds in datasplits:
    ds_seed=ds+4
    model_path=f"{MODEL_CHECKPOINTS_DIR}/ESCAPE_seq_CON_Validation_Selected_Models_CV_Datasplits_Cleaned/CV_Datasplit_{ds}"
    for model_file in os.listdir(model_path):
        model_name=model_file.removesuffix(".pt")
        print(model_name)
        model_name=model_name.replace("seed_0", f"seed_{ds_seed}")
        print(model_name)

        ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/ESCAPE_seq_CON_Validation_Selected_Models_CV_Datasplits_Cleaned/CV_Datasplit_{ds}/{model_file}"

        device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

        clear_all_gpu_memory()

        model = init_model(
            base_model_params,
            print_check=False,
            print_params=True,
        )

        state_dict = torch.load(
            ckpt_path,
            map_location="cpu",
        )

        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        model.eval()

        print("Model loaded from checkpoint...")
        print(f"Checkpoint: {ckpt_path}")
        print(f"Model device: {next(model.parameters()).device}")

        base_and_ft_models[model_name]=model

print(len(base_and_ft_models), base_and_ft_models.keys())

In [ ]:
# load cls models
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={
    "mhcprime_abt_rapid":base_model,
}

MODEL_CHECKPOINTS_DIR=PROJECT_ROOT/"Model_Checkpoints"
escape_cls_model_dir=MODEL_CHECKPOINTS_DIR/"ESCAPE_seq_CLS_Updating"

for seed in range(5):
    seed_dir=escape_cls_model_dir/f"seed_{seed}"

    for model_path in sorted(seed_dir.glob("*.pt")):
        model_name=model_path.stem

        print("Loading:",model_name)

        model=copy.deepcopy(mhcprime)
        state_dict=torch.load(model_path,map_location="cpu")
        model.load_state_dict(state_dict,strict=True)
        model.to(device)
        model.eval()

        base_and_ft_models[model_name]=model


for datasplit in range(1,5):
    split_seed=datasplit+4
    split_dir=escape_cls_model_dir/f"datasplit_{datasplit}"/"seed_0"

    for model_path in sorted(split_dir.glob("*.pt")):
        original_name=model_path.stem
        model_name=original_name.replace("_seed_0",f"_seed_{split_seed}")

        print(f"Loading: {original_name} -> {model_name}")

        model=copy.deepcopy(mhcprime)
        state_dict=torch.load(model_path,map_location="cpu")
        model.load_state_dict(state_dict,strict=True)
        model.to(device)
        model.eval()

        base_and_ft_models[model_name]=model

print("\nLoaded models:")
for model_name in base_and_ft_models:
    print(model_name)

print("\nTotal models:",len(base_and_ft_models))

#### Inference

In [ ]:
# set models
base_model=copy.deepcopy(mhcprime)

# base_and_ft_models={

#     # base model
#     # "mhcprime": base_model,

#     # updated models
#     # "escape_seq_neoant_ft_cls": escape_seq_neoant_ft_cls,
#     # "escape_seq_neoant_pepcl_cls": escape_seq_neoant_pepcl_cls,
#     # "escape_seq_neoant_ft_con": escape_seq_neoant_ft_con,
#     # "escape_seq_neoant_pepcl_con": escape_seq_neoant_pepcl_con,

#     # "escape_seq_neoant_ft_w_more_negs_cls": escape_seq_neoant_ft_w_more_negs_cls,
#     # "escape_seq_neoant_pepcl_w_more_negs_cls": escape_seq_neoant_pepcl_w_more_negs_cls,


#     # run ablated base model
#     # "mhcprime_abt": base_model, 

#     # run optimal ft and pepcl models (sept. 2026)
#     # "escape_seq_neoant_ft_cls_opt": escape_seq_neoant_ft_cls,
#     # "escape_seq_neoant_pepcl_cls_opt": escape_seq_neoant_pepcl_cls,
#     # "escape_seq_neoant_ft_con_opt": escape_seq_neoant_ft_con,
#     # "escape_seq_neoant_pepcl_con_opt": escape_seq_neoant_pepcl_con,

#     # "escape_seq_neoant_ft_w_more_negs_cls_opt": escape_seq_neoant_ft_w_more_negs_cls,
#     # "escape_seq_neoant_pepcl_w_more_negs_cls_opt": escape_seq_neoant_pepcl_w_more_negs_cls,

#     # "mhcprime_abt_rapid": base_model, 
# }


In [ ]:
clear_all_gpu_memory()

All GPU memory has been cleared.
Current GPU memory allocated: 3.97 GB
Current GPU memory reserved: 4.19 GB


In [ ]:
%%time
# build cache for background set
build_caches=False # already built
if build_caches:
    data_to_run_path=f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.pt"
    build_token_cache(chp_background_10k_per_allele_x_len_processed, data_to_run_path, tokenizer, use_domain_id=False)

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 10.3 µs


In [ ]:
%%time
"""
Score background
"""
# chp background
data_to_run_path=f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.pt"

# run_bg=False
run_bg=True

if run_bg:
    for model_name, model in base_and_ft_models.items():
        print(model_name)
        model.eval()
        with torch.no_grad():
            num_workers=16
            output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=num_workers, compile_model=False, use_domain_id=False)

        chp_background_10k_per_allele_x_len_processed[model_name]=output_scores.detach().cpu().tolist() 

mhcprime_abt
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 4484/4484 [09:03<00:00,  8.24it/s]


escape_seq_neoant_ft_cls_opt
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 4484/4484 [09:03<00:00,  8.25it/s]


escape_seq_neoant_pepcl_cls_opt
Getting data
Loading data
Setting up model
Running inference


...

escape_seq_neoant_ft_w_more_negs_cls_opt
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 4484/4484 [09:02<00:00,  8.26it/s]


escape_seq_neoant_pepcl_w_more_negs_cls_opt
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 4484/4484 [09:03<00:00,  8.25it/s]


CPU times: user 1h 3min 56s, sys: 2min 43s, total: 1h 6min 40s
Wall time: 1h 4min 8s


In [ ]:
%%time
"""
Save ranking DF with scores
"""
# chp_background_10k_per_allele_x_len_processed.to_csv(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.csv.gz")

# with optimal models
# chp_background_10k_per_allele_x_len_processed.to_csv(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed_opt.csv.gz")

CPU times: user 23min 21s, sys: 4.23 s, total: 23min 25s
Wall time: 23min 24s


In [ ]:
%%time
# rapid save
# chp_background_10k_per_allele_x_len_processed.to_pickle(
#     f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed_escape_opt.pkl",
#     protocol=5,
#     compression=None,
# )

CPU times: user 10.1 s, sys: 9.37 s, total: 19.4 s
Wall time: 19.4 s


In [ ]:
# add base model (if not already added)
# base_and_ft_models["mhcprime"]=base_model

# ablated version
base_and_ft_models["mhcprime_abt"]=base_model

In [ ]:
%%time
# escapeseq_reported_neoantigens_expanded
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(escapeseq_reported_neoantigens_expanded, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    escapeseq_reported_neoantigens_expanded[model_name]=output_scores.detach().cpu().tolist()

225it [00:00, 16445.10it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (0.1 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:02<00:00,  2.50s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:03<00:00,  3.77s/it]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:03<00:00,  3.99s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:02<00:00,  2.36s/it]


CPU times: user 2min 43s, sys: 2min 15s, total: 4min 59s
Wall time: 2min 38s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
"""
Save reported neoantigens df
"""
# escapeseq_reported_neoantigens_expanded.to_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded.csv")

# for version with optimal models
# escapeseq_reported_neoantigens_expanded.to_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded_opt.csv")

In [ ]:
%%time
# get rankings
escapeseq_reported_neoantigens_expanded["seq_len"]=escapeseq_reported_neoantigens_expanded.seq.str.replace("$", "").apply(len)
# score_cols=["mhcprime", "escape_seq_neoant_ft_cls", "escape_seq_neoant_pepcl_cls", "escape_seq_neoant_ft_con", "escape_seq_neoant_pepcl_con"]
# score_cols+=["escape_seq_neoant_ft_w_more_negs_cls", "escape_seq_neoant_pepcl_w_more_negs_cls"]


# use ablated mhcprime and optimal versions for all models
# score_cols=["mhcprime_abt", "escape_seq_neoant_ft_cls_opt", "escape_seq_neoant_pepcl_cls_opt", "escape_seq_neoant_ft_con_opt", "escape_seq_neoant_pepcl_con_opt"]
# score_cols+=["escape_seq_neoant_ft_w_more_negs_cls_opt", "escape_seq_neoant_pepcl_w_more_negs_cls_opt"]
# score_cols+=["mhcprime_abt_rapid"]

score_cols=list(base_and_ft_models.keys())
print(score_cols)

escapeseq_reported_neoantigens_expanded = add_background_percentile_ranks(
    test_df=escapeseq_reported_neoantigens_expanded,
    background_df=chp_background_10k_per_allele_x_len_processed,
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)

['mhcprime_abt_rapid', 'escape_seq_neoant_ft_cls_seed_0', 'escape_seq_neoant_ft_w_more_negs_cls_seed_0', 'escape_seq_neoant_pepcl_cls_seed_0', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_0', 'escape_seq_neoant_ft_cls_seed_1', 'escape_seq_neoant_ft_w_more_negs_cls_seed_1', 'escape_seq_neoant_pepcl_cls_seed_1', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_1', 'escape_seq_neoant_ft_cls_seed_2', 'escape_seq_neoant_ft_w_more_negs_cls_seed_2', 'escape_seq_neoant_pepcl_cls_seed_2', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_2', 'escape_seq_neoant_ft_cls_seed_3', 'escape_seq_neoant_ft_w_more_negs_cls_seed_3', 'escape_seq_neoant_pepcl_cls_seed_3', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_3', 'escape_seq_neoant_ft_cls_seed_4', 'escape_seq_neoant_ft_w_more_negs_cls_seed_4', 'escape_seq_neoant_pepcl_cls_seed_4', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_4', 'escape_seq_neoant_ft_cls_seed_5', 'escape_seq_neoant_ft_w_more_negs_cls_seed_5', 'escape_seq_neoant_pepcl_cls_seed_5', 'escape

Ranking score columns: 100%|██████████| 37/37 [00:09<00:00,  3.81it/s]

CPU times: user 7.03 s, sys: 2.95 s, total: 9.98 s
Wall time: 9.92 s


In [ ]:
"""
Save reported neoantigens df
"""
# escapeseq_reported_neoantigens_expanded.to_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded.csv")

# for version with optimal models
# escapeseq_reported_neoantigens_expanded.to_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded_opt.csv")

In [ ]:
# set models
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={

    # base model
    # "mhcprime": base_model,

    # updated models
    # "escape_seq_neoant_ft_cls": escape_seq_neoant_ft_cls,
    # "escape_seq_neoant_pepcl_cls": escape_seq_neoant_pepcl_cls,
    # "escape_seq_neoant_ft_con": escape_seq_neoant_ft_con,
    # "escape_seq_neoant_pepcl_con": escape_seq_neoant_pepcl_con,

    # "escape_seq_neoant_ft_w_more_negs_cls": escape_seq_neoant_ft_w_more_negs_cls,
    # "escape_seq_neoant_pepcl_w_more_negs_cls": escape_seq_neoant_pepcl_w_more_negs_cls,


    # run ablated base model
    "mhcprime_abt": base_model, 

    # run optimal ft and pepcl models (sept. 2026)
    "escape_seq_neoant_ft_cls_opt": escape_seq_neoant_ft_cls,
    "escape_seq_neoant_pepcl_cls_opt": escape_seq_neoant_pepcl_cls,
    "escape_seq_neoant_ft_con_opt": escape_seq_neoant_ft_con,
    "escape_seq_neoant_pepcl_con_opt": escape_seq_neoant_pepcl_con,

    "escape_seq_neoant_ft_w_more_negs_cls_opt": escape_seq_neoant_ft_w_more_negs_cls,
    "escape_seq_neoant_pepcl_w_more_negs_cls_opt": escape_seq_neoant_pepcl_w_more_negs_cls,

}

In [ ]:
%%time
# ligand_mhc_atlas_cancer_test_set_for_escape_seq
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ligand_mhc_atlas_cancer_test_set_for_escape_seq, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ligand_mhc_atlas_cancer_test_set_for_escape_seq[model_name]=output_scores.detach().cpu().tolist()

29674it [00:01, 21034.64it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (8.1 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 10/10 [00:04<00:00,  2.25it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 10/10 [00:05<00:00,  1.98it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 10/10 [00:05<00:00,  1.75it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 10/10 [00:05<00:00,  1.78it/s]


CPU times: user 20min 20s, sys: 2min 38s, total: 22min 58s
Wall time: 3min 43s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
"""
Save
"""
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
# ligand_mhc_atlas_cancer_test_set_for_escape_seq.to_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq.csv")

# for version with optimal models
# ligand_mhc_atlas_cancer_test_set_for_escape_seq.to_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq_opt.csv")

In [ ]:
%%time
# get ranks
ligand_mhc_atlas_cancer_test_set_for_escape_seq["seq_len"]=ligand_mhc_atlas_cancer_test_set_for_escape_seq.seq.str.replace("$", "").apply(len)

# score_cols=["mhcprime", "escape_seq_neoant_ft_cls", "escape_seq_neoant_pepcl_cls", "escape_seq_neoant_ft_con", "escape_seq_neoant_pepcl_con"]
# score_cols+=["escape_seq_neoant_ft_w_more_negs_cls", "escape_seq_neoant_pepcl_w_more_negs_cls"]

# use ablated mhcprime and optimal versions for all models
# score_cols=["mhcprime_abt", "escape_seq_neoant_ft_cls_opt", "escape_seq_neoant_pepcl_cls_opt", "escape_seq_neoant_ft_con_opt", "escape_seq_neoant_pepcl_con_opt"]
# score_cols+=["escape_seq_neoant_ft_w_more_negs_cls_opt", "escape_seq_neoant_pepcl_w_more_negs_cls_opt"]

score_cols=list(base_and_ft_models.keys())
print(score_cols)

ligand_mhc_atlas_cancer_test_set_for_escape_seq = add_background_percentile_ranks(
    test_df=ligand_mhc_atlas_cancer_test_set_for_escape_seq,
    background_df=chp_background_10k_per_allele_x_len_processed,
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)


['mhcprime_abt_rapid', 'escape_seq_neoant_ft_cls_seed_0', 'escape_seq_neoant_ft_w_more_negs_cls_seed_0', 'escape_seq_neoant_pepcl_cls_seed_0', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_0', 'escape_seq_neoant_ft_cls_seed_1', 'escape_seq_neoant_ft_w_more_negs_cls_seed_1', 'escape_seq_neoant_pepcl_cls_seed_1', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_1', 'escape_seq_neoant_ft_cls_seed_2', 'escape_seq_neoant_ft_w_more_negs_cls_seed_2', 'escape_seq_neoant_pepcl_cls_seed_2', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_2', 'escape_seq_neoant_ft_cls_seed_3', 'escape_seq_neoant_ft_w_more_negs_cls_seed_3', 'escape_seq_neoant_pepcl_cls_seed_3', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_3', 'escape_seq_neoant_ft_cls_seed_4', 'escape_seq_neoant_ft_w_more_negs_cls_seed_4', 'escape_seq_neoant_pepcl_cls_seed_4', 'escape_seq_neoant_pepcl_w_more_negs_cls_seed_4', 'escape_seq_neoant_ft_cls_seed_5', 'escape_seq_neoant_ft_w_more_negs_cls_seed_5', 'escape_seq_neoant_pepcl_cls_seed_5', 'escape

Ranking score columns: 100%|██████████| 37/37 [00:09<00:00,  4.09it/s]

CPU times: user 6.95 s, sys: 2.49 s, total: 9.44 s
Wall time: 9.39 s


In [ ]:
"""
Save
"""
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
# ligand_mhc_atlas_cancer_test_set_for_escape_seq.to_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq.csv")

# for version with optimal models
# ligand_mhc_atlas_cancer_test_set_for_escape_seq.to_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq_opt.csv")

### Differential mutation results inference

In [ ]:
%%time
# process gene holdout results
# central_and_model_dir_name="ESCAPE_seq_gene_holdout_experiment"

# central_and_model_dir_name="ESCAPE_seq_gene_holdout_experiment_for_ablations_model" # NOTE: must change the base model loaded in

central_and_model_dir_name="ESCAPE_seq_gene_holdout_experiment_for_ablations_model_optimal" # NOTE: must change the base model loaded in

central_ft_dir=f"{DATA_DIR}/{central_and_model_dir_name}"
output_df_dir="inference_dfs"
output_df_dir_post_inference="inference_dfs_updated"
os.makedirs(f"{central_ft_dir}/{output_df_dir_post_inference}", exist_ok=True)

genes_to_run=['JAK2', 'MAX', 'MPL', 'MED12', 'TP53', 'SRSF2', 'NOTCH2', 'EZH2', 'AKT1', 'IDH1', 'ATM', 'MYD88', 'EGFR', 'FGFR3', 'PTEN', 'SF3B1', 'CREBBP', 'MYC', 'FBXW7', 'BRAF', 'RET', 'SMAD4', 'ERBB2', 'GNAS', 'DNMT3A', 'ALK', 'KIAA1549', 'EWSR1']

# add genes from gene groups
gene_groups={
    "KRAS": ["KRAS", "KRAS-NRAS", "KRAS-KRAS-NRAS-NRAS-HRAS-HRAS", "KRAS-HRAS", "KRAS-NRAS-HRAS", "HRAS", "NRAS-HRAS"], 
    "FLT3": ["FLT3", "KIT", "PDGFRA", "FLT3-KIT-PDGFRA", "FLT3-PDGFRA"],
    "IDH2": ["IDH2", "IDH2-IDH2"],
    "PIK3CA": ["PIK3CA", "PIK3CA-PIK3CA"],
    "CTNNB1": ["CTNNB1", "CTNNB1-CTNNB1"],
}

genes_to_run+=list(gene_groups.keys())

allele_ap_matrix_wt_neg_mut_pos=pd.DataFrame()
allele_ap_matrix_wt_pos_mut_neg=pd.DataFrame()

full_spearmans=pd.DataFrame()
wt_spearmans=pd.DataFrame()
mut_spearmans=pd.DataFrame()
wt_mut_gene_diff_noX_spearmans=pd.DataFrame()

for gene in tqdm(genes_to_run):

    mut_wt_df_gene=pd.read_csv(f"{central_ft_dir}/{output_df_dir}/mut_wt_df_gene_{gene}.csv")

    mut_wt_df_gene["name_allele"]=[f"{n}_{a}" for n, a in zip(mut_wt_df_gene.name2, mut_wt_df_gene.allele)]

    model_name_ft=f"model_ft_{gene}"
    model_name_pepcl=f"model_pepcl_{gene}"    

    # inference
    escape_seq_model_dir_ft = MODEL_CHECKPOINTS_DIR / central_and_model_dir_name / "FT"
    escape_seq_model_dir_pepcl = MODEL_CHECKPOINTS_DIR / central_and_model_dir_name / "PepCL"

    model_ft=init_model(base_model_params, print_check=False, print_params=False)
    model_ft.load_state_dict(torch.load(f"{escape_seq_model_dir_ft}/{model_name_ft}.pt"))

    model_pepcl=init_model(base_model_params, print_check=False, print_params=False)
    model_pepcl.load_state_dict(torch.load(f"{escape_seq_model_dir_pepcl}/{model_name_pepcl}.pt"))

    base_model=copy.deepcopy(mhcprime)
    base_and_ft_models={
        "model_base": base_model,
        "model_ft": model_ft,
        "model_pepcl": model_pepcl,
    }

    # mut_wt_df_gene
    build_caches=True

    if build_caches:
        data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

        build_token_cache(mut_wt_df_gene, data_to_run_path, tokenizer, use_domain_id=False)

    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
    for model_name, model in base_and_ft_models.items():
        model.eval()
        with torch.no_grad():
            output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=2, compile_model=False, use_domain_id=False)

        mut_wt_df_gene[model_name]=output_scores.detach().cpu().tolist()

    mut_wt_df_gene.to_csv(f"{central_ft_dir}/{output_df_dir_post_inference}/mut_wt_df_gene_{gene}.csv")

# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

  0%|          | 0/33 [00:00<?, ?it/s]

[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']


6300it [00:00, 24671.62it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (1.7 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 3/3 [00:00<00:00,  3.58it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 2/2 [00:01<00:00,  1.06it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 33/33 [03:49<00:00,  6.96s/it]

CPU times: user 12min 56s, sys: 1min 24s, total: 14min 21s
Wall time: 3min 49s
